In [26]:
!pip install torch torchvision torchaudio
!pip install numpy pandas scikit-learn tqdm

In [27]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

In [28]:
import torch
import torch.nn as nn

class TemporalEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()

        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.output_dim = hidden_dim * 2

    def forward(self, x):
        out, _ = self.gru(x)
        return out  # [B, T, 2H]

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x, mask=None):
        # x: [B, T, H]

        scores = self.attn(x).squeeze(-1)  # [B, T]

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = F.softmax(scores, dim=1)  # [B, T]

        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)  # [B, H]

        return pooled, weights

In [30]:
class ClassifierHead(nn.Module):
    def __init__(self, input_dim, dropout=0.3):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [31]:
class AccentModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()

        self.encoder = TemporalEncoder(input_dim, hidden_dim)
        self.pooling = AttentionPooling(self.encoder.output_dim)
        self.classifier = ClassifierHead(self.encoder.output_dim)

    def forward(self, x, mask=None):
        seq_features = self.encoder(x)
        pooled, attn_weights = self.pooling(seq_features, mask)
        logits = self.classifier(pooled)

        return logits, attn_weights

In [32]:
class APLLoss(nn.Module):
    def __init__(self, lambda_apl=0.1):
        super().__init__()
        self.lambda_apl = lambda_apl
        self.eps = 1e-6

    def forward(self, logits, targets):
        # logits: [B]
        # targets: [B] (0 or 1)

        probs = torch.sigmoid(logits)

        # Cross Entropy
        ce = F.binary_cross_entropy_with_logits(logits, targets.float())

        # Reverse Cross Entropy
        probs = torch.clamp(probs, self.eps, 1 - self.eps)
        targets = torch.clamp(targets.float(), self.eps, 1 - self.eps)

        rce = -torch.mean(
            probs * torch.log(targets) +
            (1 - probs) * torch.log(1 - targets)
        )

        return ce + self.lambda_apl * rce

In [33]:
B = 4
T = 75
D = 820  # example dimension

x = torch.randn(B, T, D).to(device)
mask = torch.ones(B, T).to(device)
y = torch.randint(0, 2, (B,)).to(device)

model = AccentModel(input_dim=D).to(device)
criterion = APLLoss()

logits, attn = model(x, mask)
loss = criterion(logits, y)

print("Logits shape:", logits.shape)
print("Attention shape:", attn.shape)
print("Loss:", loss.item())

Logits shape: torch.Size([4])
Attention shape: torch.Size([4, 75])
Loss: 1.3657894134521484


In [34]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset

class AccentDataset(Dataset):
    def __init__(self, file_list, labels_dict, data_root):
        """
        file_list: list of file_ids (without extension)
        labels_dict: dict {file_id: label}
        data_root: directory containing .npy feature files
        """
        self.file_list = file_list
        self.labels_dict = labels_dict
        self.data_root = data_root

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_id = self.file_list[idx]

        x_path = os.path.join(self.data_root, f"{file_id}.npy")
        x = np.load(x_path)  # expected shape [T, D]

        x = torch.tensor(x, dtype=torch.float32)

        label = torch.tensor(self.labels_dict[file_id], dtype=torch.float32)

        return x, label

In [35]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    sequences, labels = zip(*batch)

    lengths = [seq.shape[0] for seq in sequences]

    padded_sequences = pad_sequence(sequences, batch_first=True)  # [B, T_max, D]

    mask = torch.zeros(padded_sequences.shape[:2])

    for i, length in enumerate(lengths):
        mask[i, :length] = 1

    labels = torch.stack(labels)

    return padded_sequences, mask, labels

In [36]:
from torch.utils.data import DataLoader
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score

class Trainer:
    def __init__(self, model, train_dataset, val_dataset, lr=1e-3, batch_size=8):
        self.model = model.to(device)
        self.criterion = APLLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

        self.train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn
        )

        self.val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_fn
        )

    def train_epoch(self):
        self.model.train()
        total_loss = 0

        for x, mask, y in tqdm(self.train_loader):
            x, mask, y = x.to(device), mask.to(device), y.to(device)

            self.optimizer.zero_grad()

            logits, _ = self.model(x, mask)
            loss = self.criterion(logits, y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()

            total_loss += loss.item()

        return total_loss / len(self.train_loader)

    def evaluate(self):
        self.model.eval()

        all_preds = []
        all_labels = []
        all_probs = []

        with torch.no_grad():
            for x, mask, y in self.val_loader:
                x, mask, y = x.to(device), mask.to(device), y.to(device)

                logits, _ = self.model(x, mask)
                probs = torch.sigmoid(logits)
                preds = (probs > 0.5).float()

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds)

        return acc, f1, all_probs

In [37]:
def train_model(trainer, epochs=3):
    for epoch in range(epochs):
        train_loss = trainer.train_epoch()
        val_acc, val_f1, val_probs = trainer.evaluate()

        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Accuracy: {val_acc:.4f}")
        print(f"Val F1 Score: {val_f1:.4f}")

        print("Validation Confidence Scores:")
        print(val_probs)

        print("-" * 40)

In [38]:
import pandas as pd

def run_inference(model, dataset, file_list, batch_size=8):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    model.eval()
    results = []

    idx = 0

    with torch.no_grad():
        for x, mask, _ in loader:
            x, mask = x.to(device), mask.to(device)

            logits, _ = model(x, mask)
            probs = torch.sigmoid(logits)

            batch_size_current = probs.size(0)

            for i in range(batch_size_current):
                results.append({
                    "file_id": file_list[idx],
                    "confidence": float(probs[i].cpu().item())
                })
                idx += 1

    return results

In [39]:
import os
import numpy as np

data_root = "/content/fake_features"
os.makedirs(data_root, exist_ok=True)

In [40]:
import random

num_samples = 10
D = 820

file_ids = []

for i in range(num_samples):
    file_id = f"file_{i:03d}"
    file_ids.append(file_id)

    T = random.randint(60, 90)
    fake_features = np.random.randn(T, D)

    np.save(os.path.join(data_root, f"{file_id}.npy"), fake_features)

print("Created files:", file_ids)

Created files: ['file_000', 'file_001', 'file_002', 'file_003', 'file_004', 'file_005', 'file_006', 'file_007', 'file_008', 'file_009']


In [41]:
labels = {fid: random.randint(0, 1) for fid in file_ids}
print(labels)

{'file_000': 0, 'file_001': 0, 'file_002': 0, 'file_003': 1, 'file_004': 0, 'file_005': 0, 'file_006': 0, 'file_007': 1, 'file_008': 0, 'file_009': 0}


In [42]:
train_files = file_ids[:8]
val_files = file_ids[8:]

train_labels = {fid: labels[fid] for fid in train_files}
val_labels = {fid: labels[fid] for fid in val_files}

In [43]:
train_dataset = AccentDataset(train_files, train_labels, data_root)
val_dataset = AccentDataset(val_files, val_labels, data_root)

In [44]:
x, y = train_dataset[0]
print("Sample shape:", x.shape)
print("Label:", y)

Sample shape: torch.Size([90, 820])
Label: tensor(0.)


In [45]:
model = AccentModel(input_dim=D)

In [46]:
trainer = Trainer(model, train_dataset, val_dataset, batch_size=4)

In [47]:
print(trainer.evaluate())

(1.0, 0.0, [np.float32(0.47712958), np.float32(0.47893152)])


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [48]:
train_model(trainer, epochs=3)

100%|██████████| 2/2 [00:00<00:00, 32.54it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1
Train Loss: 1.3376
Val Accuracy: 1.0000
Val F1 Score: 0.0000
Validation Confidence Scores:
[np.float32(0.4562358), np.float32(0.4561792)]
----------------------------------------


100%|██████████| 2/2 [00:00<00:00, 35.00it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2
Train Loss: 1.1801
Val Accuracy: 1.0000
Val F1 Score: 0.0000
Validation Confidence Scores:
[np.float32(0.41552365), np.float32(0.42155004)]
----------------------------------------


100%|██████████| 2/2 [00:00<00:00, 31.21it/s]

Epoch 3
Train Loss: 0.7857
Val Accuracy: 1.0000
Val F1 Score: 0.0000
Validation Confidence Scores:
[np.float32(0.338561), np.float32(0.35519773)]
----------------------------------------



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
